# 06 — Inondation fluviale avec CLIMADA

**Objectif** : explorer le module `RiverFlood` de CLIMADA et comparer avec notre approche Aqueduct des notebooks 01/02.

CLIMADA utilise les **mêmes données sources** (WRI Aqueduct, ISIMIP) mais avec une architecture différente et des fonctions d'impact JRC intégrées.

| Aspect | open_climate_risk (01/02) | CLIMADA RiverFlood (ce notebook) |
|--------|--------------------------|----------------------------------|
| Données hazard | Aqueduct S3 (COG streaming) | Aqueduct TIF (téléchargement local) |
| Vulnérabilité | JRC Huizinga (notre implémentation) | `ImpfRiverFlood` (JRC intégré) |
| Exposition | Portefeuille CSV custom | `Exposures` + `LitPop` |
| Métrique | EAD (intégration trapézoïdale) | EAI (fréquence × impact) |
| Scénarios | RCP 4.5/8.5, 5 GCMs | Idem |

**Zone** : France / Europe

---

**Sommaire** :
1. Chargement du hazard Aqueduct via CLIMADA
2. Fonctions d'impact JRC intégrées
3. Exposition LitPop (France)
4. Calcul d'impact et EAI
5. Scénarios climatiques (RCP 4.5/8.5)
6. Benchmark vs open_climate_risk

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import Point

from climada.entity import Exposures, ImpactFuncSet
from climada.entity.exposures.litpop import litpop as LitPop
from climada.engine import Impact

from climada_petals.hazard import RiverFlood
from climada_petals.entity.impact_funcs.river_flood import (
    ImpfRiverFlood, flood_imp_func_set
)

print("Imports OK")

## 1. Hazard — Inondation fluviale Aqueduct

`RiverFlood.from_aqueduct_tif()` télécharge les rasters Aqueduct et les charge directement comme objet Hazard CLIMADA.

**Paramètres clés** :
- `scenario` : `'historical'`, `'45'` (RCP 4.5), `'85'` (RCP 8.5)
- `target_year` : `1980`, `2030`, `2050`, `2080`
- `gcm` : `'WATCH'` (historique), ou un des 5 GCMs ISIMIP
- `countries` : code ISO3 (ex: `'FRA'`)
- `return_periods` : sous-ensemble de [2, 5, 10, 25, 50, 100, 250, 500, 1000]

> **Note** : le premier appel télécharge les rasters (~500 MB par scénario/GCM). Les appels suivants utilisent le cache local.

In [ ]:
# --- 1a. Charger le hazard historique (WATCH) pour la France ---
# ATTENTION : télécharge ~500 MB de rasters au premier appel
# Décommenter pour lancer

# rf_hist = RiverFlood.from_aqueduct_tif(
#     scenario='historical',
#     target_year=1980,
#     gcm='WATCH',
#     return_periods=[10, 25, 50, 100, 250, 500, 1000],
#     countries=['FRA'],
# )
# print(f"Hazard historique : {rf_hist.size} événements (périodes de retour)")
# print(f"Centroids : {rf_hist.centroids.size}")
# print(f"Profondeur max : {rf_hist.intensity.max():.1f} m")

print("Fonction from_aqueduct_tif prête.")
print("Décommenter ci-dessus pour télécharger les données Aqueduct France (~500 MB).")
print()
print("Scénarios disponibles :")
print("  historical + WATCH (baseline)")
print("  45 + [NorESM1-M, GFDL-ESM2M, HadGEM2-ES, IPSL-CM5A-LR, MIROC-ESM-CHEM] (RCP 4.5)")
print("  85 + mêmes GCMs (RCP 8.5)")
print("  Target years : 1980, 2030, 2050, 2080")

## 2. Fonctions d'impact JRC intégrées

CLIMADA intègre directement les courbes JRC Huizinga 2017, les mêmes que nous utilisons dans `open_climate_risk`. Elles sont organisées par **région** × **secteur**.

In [ ]:
# --- 2a. Explorer les fonctions d'impact JRC par région et secteur ---
regions = ["Africa", "Asia", "Europe", "North America", "Oceania", "South America"]
sectors = ["residential", "commercial", "industrial", "transport", "infrastructure", "agriculture"]

# Courbes européennes pour tous les secteurs
fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.Set2(np.linspace(0, 1, len(sectors)))

for sector, color in zip(sectors, colors):
    impf = ImpfRiverFlood.from_jrc_region_sector(region="Europe", sector=sector)
    ax.plot(impf.intensity, impf.mdd * 100, linewidth=2, color=color, label=sector)

ax.set_xlabel("Profondeur d'eau (m)")
ax.set_ylabel("Dommage relatif (%)")
ax.set_title("Fonctions d'impact JRC Huizinga 2017 — Europe")
ax.legend(title="Secteur")
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 6)
plt.tight_layout()
plt.show()

print(f"\nIntensités (profondeurs) : {impf.intensity}")
print(f"Unité : {impf.intensity_unit}")

In [ ]:
# --- 2b. Comparaison inter-régionale (résidentiel) ---
fig, ax = plt.subplots(figsize=(10, 6))
colors_reg = plt.cm.tab10(np.linspace(0, 0.6, len(regions)))

for region, color in zip(regions, colors_reg):
    impf = ImpfRiverFlood.from_jrc_region_sector(region=region, sector="residential")
    ax.plot(impf.intensity, impf.mdd * 100, linewidth=2, color=color, label=region)

ax.set_xlabel("Profondeur d'eau (m)")
ax.set_ylabel("Dommage relatif (%)")
ax.set_title("Fonctions d'impact JRC — Résidentiel, par région")
ax.legend(title="Région")
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 6)
plt.tight_layout()
plt.show()

print("L'Asie et l'Afrique ont des courbes plus basses → construction plus résiliente")
print("ou valeurs d'actifs plus faibles par rapport aux dommages maximaux.")

## 3. Exposition — LitPop et portefeuille custom

On utilise deux types d'exposition :
1. **LitPop** : exposition automatique par pays (PIB × nightlights)
2. **Portefeuille custom** : actifs spécifiques en zone inondable

In [ ]:
# --- 3a. LitPop France (résolution réduite pour la rapidité) ---
# Décommenter pour télécharger (~30s à 150 arcsec)

# litpop_fra = LitPop.LitPop.from_countries(
#     countries=['FRA'],
#     res_arcsec=150,      # ~5 km (rapide). Utiliser 30 pour ~1 km
#     fin_mode='gdp',
# )
# litpop_fra.gdf["impf_RF"] = 1  # Assigner la fonction d'impact résidentielle Europe
# litpop_fra.check()
# print(f"LitPop France : {len(litpop_fra.gdf)} points")
# print(f"Valeur totale : {litpop_fra.gdf['value'].sum()/1e9:.0f} Md$")
# litpop_fra.plot_hexbin()
# plt.title("LitPop France — distribution spatiale des actifs")
# plt.show()

print("LitPop France : décommenter pour télécharger.")
print("Résolutions : 150 arcsec (~5km, rapide) ou 30 arcsec (~1km, détaillé)")

In [ ]:
# --- 3b. Portefeuille custom (actifs en zone inondable française) ---
assets_fr = [
    {"name": "Entrepôt Arles",       "value": 8e6,   "sector": "industrial"},
    {"name": "Immeuble Paris 12e",   "value": 25e6,  "sector": "residential"},
    {"name": "Usine Strasbourg",     "value": 45e6,  "sector": "industrial"},
    {"name": "Centre comm. Nîmes",   "value": 15e6,  "sector": "commercial"},
    {"name": "Ferme Camargue",       "value": 2e6,   "sector": "agriculture"},
    {"name": "Hôpital Orléans",      "value": 60e6,  "sector": "infrastructure"},
    {"name": "Gare Bordeaux",        "value": 35e6,  "sector": "transport"},
]
coords_fr = [
    (4.63, 43.68),    # Arles
    (2.39, 48.84),    # Paris 12e
    (7.75, 48.58),    # Strasbourg
    (4.36, 43.84),    # Nîmes
    (4.50, 43.50),    # Camargue
    (1.91, 47.90),    # Orléans
    (-0.56, 44.83),   # Bordeaux
]

exp_gdf = gpd.GeoDataFrame(assets_fr, geometry=[Point(*c) for c in coords_fr])
exp_gdf["impf_RF"] = 1  # On assigne la fonction Europe résidentielle par défaut

exposure_fr = Exposures(exp_gdf)
exposure_fr.check()

print(f"Portefeuille : {len(exposure_fr.gdf)} actifs, {exposure_fr.gdf['value'].sum()/1e6:.0f} M€")
display_df = exposure_fr.gdf[["name", "value", "sector"]].copy()
display_df["value_M€"] = display_df["value"] / 1e6
display_df[["name", "sector", "value_M€"]]

## 4. Calcul d'impact

Pipeline complet : hazard + exposition + impact function → EAI.

> **Note** : cette section nécessite le téléchargement des données Aqueduct (section 1). Le code est prêt à tourner une fois les données disponibles.

In [ ]:
# --- 4a. Impact function set pour l'Europe ---
# flood_imp_func_set() retourne un set avec les 6 régions
impf_set_all = flood_imp_func_set(sector="residential")
print(impf_set_all)

# Ou une seule fonction pour l'Europe résidentielle
impf_europe_res = ImpfRiverFlood.from_jrc_region_sector("Europe", "residential")
impf_set = ImpactFuncSet([impf_europe_res])
print(f"\nFonction utilisée : {impf_europe_res.name}")
print(f"  ID : {impf_europe_res.id}")
print(f"  Profondeurs : {impf_europe_res.intensity}")
print(f"  Dommages : {np.round(impf_europe_res.mdd * 100, 1)}%")

In [ ]:
# --- 4b. Calcul d'impact (décommenter quand hazard téléchargé) ---

# impact_hist = Impact()
# impact_hist.calc(exposures=exposure_fr, impact_funcs=impf_set, hazard=rf_hist)
# print(f"EAI historique : {impact_hist.aai_agg/1e6:.2f} M€/an")
#
# # Par actif
# for i, row in exposure_fr.gdf.iterrows():
#     eai = impact_hist.eai_exp[i]
#     print(f"  {row['name']:25s} → {eai/1e3:>8.1f} k€/an  ({eai/row['value']*100:.2f}%)")

print("Pipeline d'impact prêt — décommenter quand les données Aqueduct sont téléchargées.")

## 5. Scénarios climatiques (RCP 4.5 / 8.5)

Code prêt pour comparer historique vs projections.

In [ ]:
# --- 5. Multi-scénario (décommenter quand données disponibles) ---

# scenarios = [
#     {"label": "Historique", "scenario": "historical", "year": 1980, "gcm": "WATCH",
#      "color": "#1565C0"},
#     {"label": "RCP 4.5 — 2050 (IPSL)", "scenario": "45", "year": 2050,
#      "gcm": "IPSL-CM5A-LR", "color": "#FF9800"},
#     {"label": "RCP 8.5 — 2050 (IPSL)", "scenario": "85", "year": 2050,
#      "gcm": "IPSL-CM5A-LR", "color": "#e63946"},
# ]
#
# results = {}
# for sc in scenarios:
#     rf = RiverFlood.from_aqueduct_tif(
#         scenario=sc["scenario"], target_year=sc["year"], gcm=sc["gcm"],
#         return_periods=[10, 25, 50, 100, 250, 500, 1000],
#         countries=['FRA'],
#     )
#     imp = Impact()
#     imp.calc(exposures=exposure_fr, impact_funcs=impf_set, hazard=rf)
#     results[sc["label"]] = {"impact": imp, "color": sc["color"]}
#     print(f"{sc['label']:30s} → EAI = {imp.aai_agg/1e6:.2f} M€/an")
#
# # Comparaison graphique
# fig, ax = plt.subplots(figsize=(10, 5))
# rp = np.array([10, 25, 50, 100, 250])
# for label, res in results.items():
#     fc = res["impact"].calc_freq_curve(return_per=rp)
#     ax.plot(fc.return_per, fc.impact/1e6, 'o-', color=res["color"],
#             linewidth=2, label=label)
# ax.set_xlabel("Période de retour (années)")
# ax.set_ylabel("Perte (M€)")
# ax.set_title("Inondation fluviale France — Impact par scénario climatique")
# ax.set_xscale("log")
# ax.legend()
# ax.grid(True, alpha=0.3)
# plt.tight_layout()
# plt.show()

print("Pipeline multi-scénario prêt.")

## 6. Récapitulatif — CLIMADA vs open_climate_risk

| Aspect | open_climate_risk | CLIMADA RiverFlood |
|--------|-------------------|--------------------|
| **Accès données** | S3 COG streaming (pas de téléchargement) | Téléchargement local (~500 MB/scénario) |
| **Fonctions d'impact** | Implémentation custom JRC | JRC intégré, 6 régions × 6 secteurs |
| **Exposition** | CSV portefeuille | Portefeuille + LitPop national |
| **Métrique** | EAD (intégration trapézoïdale) | EAI (fréquence × impact) |
| **Vitesse** | Rapide (streaming) | Lent au 1er appel (téléchargement) |
| **Flexibilité** | Code custom, contrôle total | Framework structuré, moins flexible |
| **Écosystème** | Standalone | Multi-péril, adaptation, UQ |

### Recommandation
- **Screening rapide** d'un actif → `open_climate_risk` (streaming S3, pas de téléchargement)
- **Analyse de portefeuille** nationale → CLIMADA `RiverFlood` + `LitPop`
- **Multi-péril** → CLIMADA (combine flood + TC + drought dans le même framework)
- **Recherche / PhD** → CLIMADA (UQ, adaptation, publications)

### Prochaines étapes
- [ ] Télécharger les données Aqueduct France et lancer le pipeline complet
- [ ] Comparer les EAD/EAI entre les deux approches sur les mêmes actifs
- [ ] Tester `from_isimip_nc()` pour les données ISIMIP brutes
- [ ] Explorer l'ensemble multi-GCM (5 modèles × 2 scénarios)